In [ ]:
import torch

torch.manual_seed(42)

# ── 0. Dati di esempio ────────────────────────────────────────────────────────
# B=2 clienti, S=4 transazioni per sequenza

amount = torch.tensor([
    [120.5,  45.0,   0.0,   0.0],   # cliente 0: 2 reali + 2 padding
    [ 33.2, 210.0,  89.5,  15.0],   # cliente 1: 4 reali
])

mcc = torch.tensor([
    [4, 7, 0, 0],
    [2, 9, 4, 1],
], dtype=torch.long)

padding_mask = torch.tensor([
    [False, False, True,  True ],   # True = posizione di padding
    [False, False, False, False],
])

batch = {
    "amount":       amount,
    "mcc":          mcc.float(),    # float per rand_like
    "padding_mask": padding_mask,
}

In [ ]:
batch

In [ ]:

print("=" * 60)
print("BATCH ORIGINALE")
print("=" * 60)
print(f"amount:\n{batch['amount']}\n")
print(f"mcc:\n{batch['mcc']}\n")
print(f"padding_mask (True=padding):\n{padding_mask}\n")


In [ ]:


# ── 1. Funzione (semplificata, senza FeatureSpec) ─────────────────────────────
def build_mtm_targets(batch, feature_names, mask_prob=0.15):
    pad_mask = batch.get("padding_mask")
    targets  = {}
    masks    = {}

    for name in feature_names:
        # ① Salva copia dei valori originali come target
        targets[name] = batch[name].clone()

        # ② Campionamento bernoulliano: ogni posizione → True con prob mask_prob
        rand = torch.rand_like(batch[name], dtype=torch.float32)
        m = rand < mask_prob

        print(f"\n{'─'*60}")
        print(f"Feature: '{name}'")
        print(f"  rand values:\n{rand.round(decimals=2)}")
        print(f"  m_raw (rand < {mask_prob}):\n{m}")

        # ③ Annulla il masking sulle posizioni di padding
        if pad_mask is not None:
            m = m & ~pad_mask
            print(f"  m dopo annullamento padding:\n{m}")

        masks[name] = m

        # ④ Sostituisce le posizioni mascherate con 0 nel batch
        batch[name] = batch[name].masked_fill(m, 0)
        print(f"  batch['{name}'] dopo masked_fill:\n{batch[name]}")

    return targets, masks


# ── 2. Esecuzione ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("ESECUZIONE build_mtm_targets  (mask_prob=0.15)")
print("=" * 60)

feature_names = ["amount", "mcc"]
targets, masks = build_mtm_targets(batch, feature_names, mask_prob=0.15)


# ── 3. Riepilogo finale ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("RIEPILOGO FINALE")
print("=" * 60)

for name in feature_names:
    print(f"\n── {name} ──")
    print(f"  targets  (valori originali) :\n{targets[name]}")
    print(f"  masks    (True=mascherato)  :\n{masks[name]}")
    print(f"  batch    (input al modello) :\n{batch[name]}")


# ── 4. Posizioni supervisionate dalla MTM loss ────────────────────────────────
print("\n" + "=" * 60)
print("POSIZIONI SUPERVISIONATE DALLA MTM LOSS")
print("=" * 60)

for name in feature_names:
    idxs = masks[name].nonzero(as_tuple=False)
    if len(idxs) == 0:
        print(f"\n'{name}': nessuna posizione mascherata (sfortuna statistica!)")
        continue
    print(f"\n'{name}':")
    for b, s in idxs:
        valore_target = targets[name][b, s].item()
        valore_input  = batch[name][b, s].item()
        print(f"  cliente={b.item()} pos={s.item()} │ "
              f"il modello vede {valore_input:.1f} → deve predire {valore_target:.4g}")

# DATASET

In [ ]:
import os 
import sys
os.chdir("..")

In [ ]:
from src.data import load_dataframe, TransactionDataset
from pathlib import Path
import pandas as pd


In [ ]:
from src.model import NumericFeature, HighCardCategoricalFeature, CategoricalFeature, DatetimeFeature

In [ ]:
data = Path("data")

In [ ]:
sorted(data.glob("*.csv"))

In [ ]:
df = load_dataframe(Path("data") / "transactions.csv")


In [ ]:
df

In [ ]:
features = [
        NumericFeature("importo", signed=True),
        NumericFeature("saldo_post"),
        NumericFeature("delta_t"),
        HighCardCategoricalFeature("merchant"),
        CategoricalFeature("mcc",        801),
        CategoricalFeature("canale",      11),
        CategoricalFeature("macro_tipo",   9),
        CategoricalFeature("sotto_tipo",  41),
        CategoricalFeature("divisa",       6),
        DatetimeFeature("timestamp"),
    ]

In [ ]:
features[0]

In [ ]:
features[0].n_slots, features[0].name

In [ ]:
delta_t_values = (
        df.sort_values(["client_id", "timestamp"])
          .groupby("client_id")["timestamp"]
          .diff()
          .dropna()
          .clip(lower=0)
          .to_numpy()
    )

In [ ]:
for feat in features:
    if isinstance(feat, NumericFeature):
        if feat.name == "delta_t":
            feat.fit(delta_t_values)
        elif feat.name in df.columns:
            feat.fit(df[feat.name].to_numpy())

In [ ]:
ds = TransactionDataset(df, seq_len=32, windows_per_client=4)

In [ ]:
ds

# DATASET

In [6]:
from torch.utils.data import Dataset, SequentialSampler, RandomSampler, WeightedRandomSampler
import torch

In [2]:
torch.arange(10, dtype=torch.float32)

tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])

In [4]:
class NumbersDataset(Dataset):
    # 10 sample: ogni sample è un vettore [x, x²]
    def __init__(self):
        self.data = torch.arange(10, dtype=torch.float32)

    def __len__(self):
        return 10

    def __getitem__(self, idx):
        x = self.data[idx]              # es. idx=3 → x=3.0
        return torch.stack([x, x**2])   # → tensor([3., 9.])

ds = NumbersDataset()
print(ds[0])   # → tensor([0., 0.])
print(ds[3])   # → tensor([3., 9.])
print(ds[9])   # → tensor([9., 81.])

tensor([0., 0.])
tensor([3., 9.])
tensor([ 9., 81.])


In [5]:
len(ds)

10

In [8]:
for i in SequentialSampler(ds):
    print(i)

0
1
2
3
4
5
6
7
8
9


In [9]:
for i in RandomSampler(ds):
    print(i)

1
5
2
6
7
9
8
0
3
4


In [7]:
list(SequentialSampler(ds))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [10]:
weights = torch.ones(10)
weights[9] = 10.0 
for i in WeightedRandomSampler(weights, num_samples=20):
    print(i)

8
9
2
4
0
3
0
7
7
9
9
0
8
9
8
2
9
5
9
9


In [ ]:
# SequentialSampler (default, shuffle=False)
list(SequentialSampler(ds))
# → [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# RandomSampler (shuffle=True)
list(RandomSampler(ds))
# → [7, 2, 5, 0, 9, 3, 1, 8, 4, 6]  (esempio)

# WeightedRandomSampler — campiona con più prob. certi idx
 # idx 9 ha 10x più probabilità
sampler = WeightedRandomSampler(weights, num_samples=10)
list(sampler)
# → [9, 9, 3, 9, 1, 9, 9, 2, 9, 5]  (esempio)

In [11]:
from torch.utils.data import DataLoader

In [21]:
loader = DataLoader(
    ds,
    batch_size=4,
    shuffle=True,
    #num_workers=2,     # 2 subprocess in parallelo
    #pin_memory=True,    # pinned memory → DMA verso GPU
    drop_last=False,    # mantieni l'ultimo batch anche se < 4
)

for batch in loader:
    # batch.shape = (4, 2)  →  4 sample, ognuno [x, x²]
    print(batch)

tensor([[ 6., 36.],
        [ 1.,  1.],
        [ 2.,  4.],
        [ 3.,  9.]])
tensor([[ 9., 81.],
        [ 8., 64.],
        [ 0.,  0.],
        [ 5., 25.]])
tensor([[ 4., 16.],
        [ 7., 49.]])


In [23]:
def my_collate(samples):
    # samples = lista di tensori shape (2,), es. [tensor([7.,49.]), tensor([2.,4.]), ...]
    stacked = torch.stack(samples)          # (B, 2)
    x  = stacked[:, 0]                      # (B,)  → i valori x
    x2 = stacked[:, 1]                      # (B,)  → i valori x²
    return {"x": x, "x_squared": x2}       # dict invece di tensor

loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=my_collate)

for batch in loader:
    print(batch["x"])         # tensor([7., 2., 5., 0.])
    print(batch["x_squared"]) # tensor([49., 4., 25., 0.])
    print("---------")

tensor([9., 6., 0., 8.])
tensor([81., 36.,  0., 64.])
---------
tensor([1., 5., 4., 3.])
tensor([ 1., 25., 16.,  9.])
---------
tensor([7., 2.])
tensor([49.,  4.])
---------
